In [1]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="ollama",
    model="qwen3:8b"
)

In [2]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    context_schema=ColourContext  
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [5]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='c9c7bf2f-3449-4e1e-95f0-826baa888956'),
              AIMessage(content='<think>\nOkay, the user asked, "What is my favourite colour?" Hmm, that\'s a bit tricky. I can\'t really know their favorite color unless they tell me. I should respond in a way that encourages them to share their preference. Let me think about how to phrase that.\n\nMaybe start by acknowledging that I can\'t know their favorite color for sure. Then, offer to help them explore different colors or discuss their preferences. I should keep the tone friendly and open-ended. Also, maybe mention that colors can have different meanings or associations, which might help them reflect on their own preferences. Let me make sure the response is positive and not too technical. Yeah, that should work. Let me put that together clearly.\n</think>\n\nI can\'t know your favorite color unless you tell me! 🌈 Would you li

## Accessing Context

In [6]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [7]:
agent = create_agent(
    model=model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [8]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

C:\Users\ASUS\miniconda3\envs\Ollama\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='7b7a74d7-3262-4179-b13d-b8000f24f80a'),
              AIMessage(content='<think>\nOkay, the user is asking, "What is my favourite colour?" Let me check the available tools. There\'s a function called get_favourite_colour that retrieves the user\'s favourite colour. Since the user is directly asking about their favourite colour, I should call that function. The parameters required are none, so the arguments object will be empty. I need to make sure to structure the tool call correctly within the XML tags. Let me verify again: the function exists, no parameters needed. Alright, time to generate the tool call.\n</think>\n\n', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-25T13:30:28.5540356Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2775283000, 'load_duration': 33888100, 'prompt_eval_count': 182, 'prompt_eval_duration': 24

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='ed883b67-1015-4144-988c-9c4ad1613899'),
              AIMessage(content='<think>\nOkay, the user is asking, "What is my favourite colour?" Let me check the available tools. There\'s a function called get_favourite_colour that retrieves the user\'s favourite colour. Since the user is directly asking about their favourite colour, I should call that function. The parameters required are none, so the arguments object will be empty. I need to make sure I use the correct JSON structure within the tool_call tags. Let me format it properly.\n</think>\n\n', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-25T13:31:21.8376048Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2571212100, 'load_duration': 29458500, 'prompt_eval_count': 182, 'prompt_eval_duration': 6390100, 'eval_count': 114, 'eval_duration': 2534358200, 'logprobs': None, 'mo

## State

In [10]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [11]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [13]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [14]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='4dbe2631-1df2-4e2b-a5eb-5e47d6c5ffb7'),
              AIMessage(content='<think>\nOkay, the user said their favorite color is green. I need to update the state with that information. Let me check the available tools. There\'s a function called update_favourite_colour that takes a favourite_colour parameter. The user has already revealed their favorite color, so I should call that function with "green" as the argument. I\'ll make sure the JSON is correctly formatted with the name and arguments.\n</think>\n\n', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-25T13:38:23.0041896Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4383799300, 'load_duration': 2076504000, 'prompt_eval_count': 157, 'prompt_eval_duration': 217654000, 'eval_count': 110, 'eval_duration': 2088171600, 'logprobs': None, 'model_na

In [15]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='99f79f5f-78f9-474e-b4bf-ea19c0678ee9'),
              AIMessage(content='<think>\nOkay, the user said, "Hello, how are you?" I need to respond appropriately. Let me check the tools provided. The only function available is update_favourite_colour, which requires a favourite_colour parameter. But the user isn\'t mentioning their favourite colour here. They\'re just greeting me and asking about my well-being. Since there\'s no mention of colour, I don\'t need to call any function. I should just reply with a friendly greeting and offer assistance. No tool call is necessary here.\n</think>\n\nHello! I\'m just a virtual assistant, but I\'m here and ready to help. How can I assist you today? 😊', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-25T13:40:16.2218528Z', 'done': True, 'done_reason': 'stop', 'total_duration': 278

## Read state

In [17]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [18]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='bf8e5201-87ad-4218-85f8-7571fc384bbc'),
              AIMessage(content='<think>\nOkay, the user said their favorite color is green. I need to update the state with that information. Let me check the available tools. There\'s the update_favourite_colour function which requires the favourite_colour parameter. Since the user provided "green" as their favorite color, I should call that function with "green" as the argument. I don\'t need to use the read function here because the user already revealed their choice. Just need to make sure the parameters are correctly formatted in JSON within the tool_call tags.\n</think>\n\n', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-25T14:03:49.3900387Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4900486700, 'load_duration': 2095193500, 'prompt_eval_count': 

In [19]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='bf8e5201-87ad-4218-85f8-7571fc384bbc'),
              AIMessage(content='<think>\nOkay, the user said their favorite color is green. I need to update the state with that information. Let me check the available tools. There\'s the update_favourite_colour function which requires the favourite_colour parameter. Since the user provided "green" as their favorite color, I should call that function with "green" as the argument. I don\'t need to use the read function here because the user already revealed their choice. Just need to make sure the parameters are correctly formatted in JSON within the tool_call tags.\n</think>\n\n', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-25T14:03:49.3900387Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4900486700, 'load_duration': 2095193500, 'prompt_eval_count': 